<a href="https://colab.research.google.com/github/tusharsingh3199/3D-Brain-Tumor-Segmentation/blob/main/BrainTumorSegemtation/Experiment%20Notebooks/3D_UNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Library, Download Data, Patient Setup, EDA**

In [1]:
import os
import zipfile
import random
import nibabel as nib
from ipywidgets import interact, IntSlider

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


In [2]:
!kaggle datasets download --d aryashah2k/brain-tumor-segmentation-brats-2019

Dataset URL: https://www.kaggle.com/datasets/aryashah2k/brain-tumor-segmentation-brats-2019
License(s): CC0-1.0
brain-tumor-segmentation-brats-2019.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
Source_File = "/content/brain-tumor-segmentation-brats-2019.zip"
Data_Path = "/content/MICCAI_BraTS_2019_Data_Training"

if not os.path.exists(Data_Path):
    os.makedirs(Data_Path)
    with zipfile.ZipFile(Source_File, 'r') as zip_ref:
        zip_ref.extractall()

def patients_dir(Data_Path):
    patient_dir = []
    for grade in ["HGG", "LGG"]:
        grade_dir = os.path.join(Data_Path, grade)
        patients = sorted(os.listdir(grade_dir))
        patient_paths = [os.path.join(grade_dir, p) for p in patients]

        for path in patient_paths:
            patient_dir.append(path)

    return patient_dir

patients_dir = patients_dir(Data_Path)


In [ ]:
def EDA(patients_dir):
    patient = random.choice(patients_dir)
    img = []
    seg = []
    for i in sorted(os.listdir(patient)):
        if "seg" in os.path.join(patient, i):
            seg = nib.load(os.path.join(patient, i)).get_fdata()
        else:
            v = nib.load(os.path.join(patient, i)).get_fdata()
            v = ((v - v.min()) / (v.max() - v.min() + 1e-8) * 255).astype(np.uint8)
            img.append(v)

    def view_slice(z):
        plt.figure(figsize=(12, 6))
        for i in range(4):
            plt.subplot(2, 4, i+1)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            plt.subplot(2, 4, 5+i)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            slice_seg = seg[:, :, z]
            masked_seg = np.ma.masked_where(slice_seg == 0, slice_seg)
            plt.imshow(masked_seg, cmap="jet", alpha=0.7, vmin=0, vmax=4)

        plt.show()

    interact(view_slice, z=IntSlider(min=0, max=150, step=1, value=75))

EDA(patients_dir)
len(patients_dir)

# Preparing the Dataset for 3D UNet, SWIN Transformer

In [5]:
train_dir, val_dir, test_dir = patients_dir[:265], patients_dir[265:300], patients_dir[300:]

def load_patient(patient_dir):
    patient_dir = patient_dir.numpy().decode()
    images = []
    for modality in ["t1", "t1ce", "t2", "flair"]:
        v = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_{modality}.nii")).get_fdata().astype(np.float32)
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        images.append(v)

    image = np.stack(images, axis=-1).astype(np.float32)
    mask = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_seg.nii")).get_fdata().astype(np.int32)
    mask[mask == 4] = 3

    patch_size = (128, 128, 128)
    tumor = np.argwhere(mask > 0)

    if len(tumor) > 0:
        cx, cy, cz = tumor[np.random.randint(len(tumor))]
        x = np.clip(cx - patch_size[0] // 2, 0, image.shape[0] - patch_size[0])
        y = np.clip(cy - patch_size[1] // 2, 0, image.shape[1] - patch_size[1])
        z = np.clip(cz - patch_size[2] // 2, 0, image.shape[2] - patch_size[2])
    else:
        x = np.random.randint(0, image.shape[0] - patch_size[0] + 1)
        y = np.random.randint(0, image.shape[1] - patch_size[1] + 1)
        z = np.random.randint(0, image.shape[2] - patch_size[2] + 1)

    image = image[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2], :]
    mask = mask[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2]]

    return image, mask

def tf_load_patient(patient_dir):
    image, mask = tf.py_function(load_patient, [patient_dir], [tf.float32, tf.int32])
    image.set_shape((240, 240, 155, 4))
    mask.set_shape((240, 240, 155))
    return image, mask


train_dataset = (tf.data.Dataset.from_tensor_slices(train_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .shuffle(len(train_dir)).batch(1).prefetch(tf.data.AUTOTUNE))

val_dataset = (tf.data.Dataset.from_tensor_slices(val_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))

test_dataset = (tf.data.Dataset.from_tensor_slices(test_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))